# Graph Embeddings Visualization (Bank Transactions)

This notebook builds a transaction graph from `bank_transaction_data.csv` and visualizes account embeddings using multiple views.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme(style="whitegrid")


In [ ]:
df = pd.read_csv("bank_transaction_data.csv")
if len(df) > 60000:
    df = df.sample(60000, random_state=SEED).reset_index(drop=True)

df["Transaction Amount"] = pd.to_numeric(df["Transaction Amount"], errors="coerce")
df["Latency (ms)"] = pd.to_numeric(df["Latency (ms)"], errors="coerce")
df["Slice Bandwidth (Mbps)"] = pd.to_numeric(df["Slice Bandwidth (Mbps)"], errors="coerce")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["Fraud Flag"] = df["Fraud Flag"].astype(str).str.lower().map({"true": 1, "false": 0})
df = df.dropna(subset=[
    "Sender Account ID", "Receiver Account ID", "Transaction Amount", "Timestamp", "Fraud Flag"
]).copy()

df["hour"] = df["Timestamp"].dt.hour
df["dayofweek"] = df["Timestamp"].dt.dayofweek

print(f"Rows: {len(df):,}")
print(f"Accounts: {pd.unique(df[['Sender Account ID','Receiver Account ID']].values.ravel()).shape[0]:,}")
df.head(2)


In [ ]:
# Build directed graph where nodes are accounts and edges are transactions.
G = nx.DiGraph()
for _, r in df.iterrows():
    s = f"A_{r['Sender Account ID']}"
    t = f"A_{r['Receiver Account ID']}"
    G.add_node(s)
    G.add_node(t)
    G.add_edge(
        s, t,
        amount=float(r["Transaction Amount"]),
        fraud=int(r["Fraud Flag"]),
        latency=float(r["Latency (ms)"] if pd.notna(r["Latency (ms)"]) else 0.0),
        bw=float(r["Slice Bandwidth (Mbps)"] if pd.notna(r["Slice Bandwidth (Mbps)"]) else 0.0),
        hour=int(r["hour"]),
    )

nodes = list(G.nodes())
print(f"Nodes: {G.number_of_nodes():,} | Edges: {G.number_of_edges():,}")


In [ ]:
# Lightweight handcrafted embedding per account.
pr = nx.pagerank(G, alpha=0.85)
undirected = G.to_undirected()

def node_embedding(n):
    out_e = list(G.out_edges(n, data=True))
    in_e = list(G.in_edges(n, data=True))
    amounts_out = [e[2]["amount"] for e in out_e] if out_e else [0.0]
    amounts_in = [e[2]["amount"] for e in in_e] if in_e else [0.0]
    fraud_vals = [e[2]["fraud"] for e in out_e + in_e] if (out_e or in_e) else [0.0]
    latency_vals = [e[2]["latency"] for e in out_e + in_e] if (out_e or in_e) else [0.0]
    bw_vals = [e[2]["bw"] for e in out_e + in_e] if (out_e or in_e) else [0.0]
    nbrs = set([v for _, v, _ in out_e] + [u for u, _, _ in in_e])
    return np.array([
        G.out_degree(n),
        G.in_degree(n),
        np.mean(amounts_out), np.std(amounts_out),
        np.mean(amounts_in), np.std(amounts_in),
        np.mean(fraud_vals),
        np.mean(latency_vals), np.std(latency_vals),
        np.mean(bw_vals), np.std(bw_vals),
        len(nbrs),
        nx.clustering(undirected, n),
        pr.get(n, 0.0)
    ], dtype=float)

X_raw = np.vstack([node_embedding(n) for n in nodes])
X = StandardScaler().fit_transform(X_raw)
print("Embedding matrix:", X.shape)


In [ ]:
# 2D projections.
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X)

tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(X)-1), init="pca")
X_tsne = tsne.fit_transform(X)

print("PCA variance explained:", pca.explained_variance_ratio_.round(3))


In [ ]:
# Visualization 1: PCA and t-SNE by total degree.
degree = np.array([G.degree(n) for n in nodes])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
s1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=degree, cmap="viridis", s=18, alpha=0.75)
axes[0].set_title("PCA projection (color = degree)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.colorbar(s1, ax=axes[0], label="Degree")

s2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=degree, cmap="plasma", s=18, alpha=0.75)
axes[1].set_title("t-SNE projection (color = degree)")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
fig.colorbar(s2, ax=axes[1], label="Degree")

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Cluster structure on t-SNE.
k = 5 if len(X) >= 5 else max(2, len(X))
kmeans = KMeans(n_clusters=k, n_init=10, random_state=SEED)
clusters = kmeans.fit_predict(X)

plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=clusters, palette="Set2", s=28, linewidth=0, alpha=0.8)
plt.title("t-SNE colored by KMeans clusters")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 3: Fraud-involvement overlay and feature diagnostics.
fraud_nodes = set()
for u, v, d in G.edges(data=True):
    if d["fraud"] == 1:
        fraud_nodes.add(u)
        fraud_nodes.add(v)
fraud_label = np.array([1 if n in fraud_nodes else 0 for n in nodes])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1], hue=fraud_label,
    palette={0: "#4C78A8", 1: "#E45756"}, s=24, alpha=0.75, ax=axes[0]
)
axes[0].set_title("PCA: fraud-involved account overlay")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(title="Fraud involved", labels=["No", "Yes"])

feature_importance = pd.Series(np.var(X, axis=0)).sort_values(ascending=False).head(8)
axes[1].barh(range(len(feature_importance)), feature_importance.values[::-1], color="#72B7B2")
axes[1].set_yticks(range(len(feature_importance)))
axes[1].set_yticklabels([f"Feature {i}" for i in feature_importance.index[::-1]])
axes[1].set_title("Top embedding dimensions by variance")
axes[1].set_xlabel("Variance")

plt.tight_layout()
plt.show()


In [ ]:
# Compact cluster summary table.
summary = pd.DataFrame({
    "cluster": clusters,
    "degree": degree,
    "fraud_involved": fraud_label,
    "pca_x": X_pca[:, 0],
    "pca_y": X_pca[:, 1],
})

cluster_summary = summary.groupby("cluster").agg(
    n_accounts=("cluster", "size"),
    avg_degree=("degree", "mean"),
    fraud_ratio=("fraud_involved", "mean"),
).reset_index().sort_values("n_accounts", ascending=False)

display(cluster_summary)


## Key takeaways

- Embeddings separate account behavior into visible regions/clusters.
- Fraud-involved accounts can form partially distinct neighborhoods.
- Degree and transaction dynamics strongly influence embedding geometry.
- These plots can be used as a baseline before moving to deeper methods (DeepWalk/Node2Vec/GNNs).
